In [0]:
# Databricks notebook source

import pandas as pd
import numpy as np
import requests
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, IntegerType

# ============================================================================
# CONFIG
# ============================================================================

# Données disponibles depuis le 3 juin 2026
START_DATE = datetime(2026, 6, 3)
END_DATE = datetime.now()

# Unity Catalog configuration
CATALOG = "workspace"
SCHEMA = "energy_forecast"
OUTPUT_TABLE_AGG = f"{CATALOG}.{SCHEMA}.gen_output_capability"
OUTPUT_TABLE_DETAIL = f"{CATALOG}.{SCHEMA}.gen_output_capability_by_generator"

# ============================================================================
# HELPERS
# ============================================================================

def fetch_gen_output_capability_daily(start_date, end_date):
    """
    Télécharge les données GenOutputCapability jour par jour.
    Chaque fichier contient les 24 heures de la journée pour tous les générateurs.
    
    Args:
        start_date: Date de début
        end_date: Date de fin
    
    Returns:
        DataFrame avec colonnes: Date, Hour, GeneratorName, FuelType, OutputMW, CapabilityMW
    """
    
    date_range = pd.date_range(start_date, end_date, freq='D')
    all_data = []
    
    print(f"\n📥 Downloading GenOutputCapability (daily files)")
    print(f"   Period: {start_date.date()} to {end_date.date()} ({len(date_range)} days)\n")
    
    for date in date_range:
        date_str = date.strftime('%Y%m%d')
        
        # Essayer d'abord le fichier sans version (dernière version du jour)
        # puis essayer les versions spécifiques si échec
        urls_to_try = [
            f"https://reports-public.ieso.ca/public/GenOutputCapability/PUB_GenOutputCapability_{date_str}.xml"
        ]
        
        # Ajouter les versions horaires (v1 à v24)
        for v in range(24, 0, -1):  # Essayer de la plus récente à la plus ancienne
            urls_to_try.append(
                f"https://reports-public.ieso.ca/public/GenOutputCapability/PUB_GenOutputCapability_{date_str}_v{v}.xml"
            )
        
        success = False
        for url in urls_to_try:
            try:
                r = requests.get(url, timeout=60)
                r.raise_for_status()
                
                # Parse XML avec namespace IMO (différent des autres fichiers IESO)
                root = ET.fromstring(r.content)
                ns = {'imo': 'http://www.theIMO.com/schema'}
                
                doc_body = root.find('imo:IMODocBody', ns)
                if doc_body is None:
                    continue
                
                # Extraire la date du document
                date_elem = doc_body.find('imo:Date', ns)
                doc_date = date_elem.text if date_elem is not None else None
                
                generators_elem = doc_body.find('imo:Generators', ns)
                if generators_elem is None:
                    continue
                
                rows = []
                
                # Parser chaque générateur
                for gen in generators_elem.findall('imo:Generator', ns):
                    gen_name_elem = gen.find('imo:GeneratorName', ns)
                    fuel_type_elem = gen.find('imo:FuelType', ns)
                    
                    gen_name = gen_name_elem.text if gen_name_elem is not None else None
                    fuel_type = fuel_type_elem.text if fuel_type_elem is not None else None
                    
                    # Extraire les données de sortie (Output)
                    outputs_elem = gen.find('imo:Outputs', ns)
                    outputs_by_hour = {}
                    if outputs_elem is not None:
                        for output in outputs_elem.findall('imo:Output', ns):
                            hour_elem = output.find('imo:Hour', ns)
                            energy_elem = output.find('imo:EnergyMW', ns)
                            
                            if hour_elem is not None and energy_elem is not None:
                                hour = int(hour_elem.text)
                                outputs_by_hour[hour] = float(energy_elem.text)
                    
                    # Extraire les données de capacité (Capability)
                    capabilities_elem = gen.find('imo:Capabilities', ns)
                    capabilities_by_hour = {}
                    if capabilities_elem is not None:
                        for capability in capabilities_elem.findall('imo:Capability', ns):
                            hour_elem = capability.find('imo:Hour', ns)
                            energy_elem = capability.find('imo:EnergyMW', ns)
                            
                            if hour_elem is not None and energy_elem is not None:
                                hour = int(hour_elem.text)
                                capabilities_by_hour[hour] = float(energy_elem.text)
                    
                    # Créer une ligne par heure (1-24)
                    for hour in range(1, 25):
                        rows.append({
                            'Date': doc_date,
                            'Hour': hour,
                            'GeneratorName': gen_name,
                            'FuelType': fuel_type,
                            'OutputMW': outputs_by_hour.get(hour),
                            'CapabilityMW': capabilities_by_hour.get(hour)
                        })
                
                if rows:
                    all_data.extend(rows)
                    print(f"   ✅ {date_str}: {len(rows)} records ({len(rows)//24} generators × 24h)")
                    success = True
                    break
                    
            except Exception as e:
                continue
        
        if not success:
            print(f"   ⚠️  {date_str}: No data available")
    
    if not all_data:
        return pd.DataFrame()
    
    df = pd.DataFrame(all_data)
    
    print(f"\n✅ Total records: {len(df):,}")
    print(f"   Generators: {df['GeneratorName'].nunique():,}")
    print(f"   Date range: {df['Date'].min()} to {df['Date'].max()}")
    
    return df


def create_ieso_datetime(date_col, hour_col):
    """
    Crée un datetime UTC à partir de colonnes Date et Hour IESO.
    IESO utilise le fuseau horaire America/Toronto (EST/EDT).
    """
    dt = (
        pd.to_datetime(date_col)
        + pd.to_timedelta(hour_col.astype(int) - 1, unit="h")
    )
    
    # Localiser en timezone Toronto puis convertir en UTC
    dt = (
        dt.dt.tz_localize(
            "America/Toronto",
            ambiguous=False,
            nonexistent="shift_forward"
        )
        .dt.tz_convert("UTC")
        .dt.tz_localize(None)
    )
    
    return dt

# ============================================================================
# FETCH DATA
# ============================================================================

print("="*80)
print("GENERATOR OUTPUT AND CAPABILITY IMPORT")
print("="*80)
print(f"\nSource: IESO GenOutputCapability Reports")
print(f"Period: {START_DATE.date()} to {END_DATE.date()}")
print(f"Target tables:")
print(f"  - Aggregated: {OUTPUT_TABLE_AGG}")
print(f"  - By generator: {OUTPUT_TABLE_DETAIL}")

df = fetch_gen_output_capability_daily(START_DATE, END_DATE)

if df.empty:
    raise Exception("No data fetched. Please check date range and URL availability.")

# ============================================================================
# TRANSFORM DATA
# ============================================================================

print("\n📊 Transforming data...")

# Créer datetime UTC
df['datetime'] = create_ieso_datetime(df['Date'], df['Hour'])

# Nettoyer les types de carburant
df['FuelType'] = df['FuelType'].astype(str).str.upper().str.strip()

# Convertir les MW en numeric (gérer les valeurs manquantes)
df['OutputMW'] = pd.to_numeric(df['OutputMW'], errors='coerce')
df['CapabilityMW'] = pd.to_numeric(df['CapabilityMW'], errors='coerce')

# Calculer le taux d'utilisation
df['UtilizationRate'] = (df['OutputMW'] / df['CapabilityMW']).replace([np.inf, -np.inf], np.nan)

print(f"   ✅ Datetime created and converted to UTC")
print(f"   ✅ Missing values: OutputMW={df['OutputMW'].isna().sum()}, CapabilityMW={df['CapabilityMW'].isna().sum()}")

# ============================================================================
# SAVE DETAILED DATA (BY GENERATOR)
# ============================================================================

print("\n💾 Saving detailed data by generator...")

# Sélectionner les colonnes pour la table détaillée
detail_df = df[['datetime', 'GeneratorName', 'FuelType', 'OutputMW', 'CapabilityMW', 'UtilizationRate']].copy()

# Convertir en Spark DataFrame
spark_detail_df = spark.createDataFrame(detail_df)

# Écrire dans Unity Catalog
spark_detail_df.write \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(OUTPUT_TABLE_DETAIL)

print(f"   ✅ Detailed data saved to {OUTPUT_TABLE_DETAIL}")
print(f"   ✅ Rows written: {spark_detail_df.count():,}")
print(f"   ✅ Generators: {detail_df['GeneratorName'].nunique():,}")

# ============================================================================
# AGGREGATE BY FUEL TYPE
# ============================================================================

print("\n📈 Aggregating by fuel type...")

agg_df = (
    df.groupby(['datetime', 'FuelType'], as_index=False)
    .agg({
        'OutputMW': 'sum',
        'CapabilityMW': 'sum',
        'GeneratorName': 'count'  # Nombre de générateurs
    })
    .rename(columns={'GeneratorName': 'NumGenerators'})
)

# Recalculer le taux d'utilisation après agrégation
agg_df['UtilizationRate'] = (agg_df['OutputMW'] / agg_df['CapabilityMW']).replace([np.inf, -np.inf], np.nan)

print(f"   ✅ Aggregated to {len(agg_df):,} rows")

# ============================================================================
# DISPLAY STATISTICS
# ============================================================================

print("\n" + "="*80)
print("DATA SUMMARY")
print("="*80)

print(f"\n📊 Overall Statistics:")
print(f"   Total records: {len(agg_df):,}")
print(f"   Date range: {agg_df['datetime'].min()} to {agg_df['datetime'].max()}")
print(f"   Unique fuel types: {agg_df['FuelType'].nunique()}")
print(f"   Fuel types: {sorted(agg_df['FuelType'].unique())}")

print(f"\n📈 Summary by Fuel Type:")
summary = (
    agg_df.groupby('FuelType')
    .agg({
        'OutputMW': ['count', 'mean', 'min', 'max', 'sum'],
        'CapabilityMW': ['mean', 'max'],
        'UtilizationRate': 'mean',
        'NumGenerators': 'mean'
    })
    .round(2)
)

display(summary)

# ============================================================================
# SAVE TO UNITY CATALOG
# ============================================================================

print("\n💾 Saving aggregated data by fuel type...")

# Convertir en Spark DataFrame
spark_df = spark.createDataFrame(agg_df)

# Écrire dans Unity Catalog (mode overwrite pour ce bronze layer)
spark_df.write \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(OUTPUT_TABLE_AGG)

print(f"   ✅ Aggregated data saved to {OUTPUT_TABLE_AGG}")
print(f"   ✅ Rows written: {spark_df.count():,}")

# ============================================================================
# CRÉER LA TABLE GENERATOR_ZONES
# ============================================================================

print("\n🗺️ Creating generator_zones table...")

# Données des générateurs et leurs zones
from io import StringIO

generator_zones_data = """GeneratorName,Zone
ATIKOKAN-G1,NORTHWEST
CALSTOCKGS,NORTHEAST
TBAYBOWATER CTS,NORTHWEST
VAUGHAN 3 BESS,TORONTO
BRIGHTON BEACH,SOUTHWEST
CARDINAL,EAST
COCHRANECGS,NORTHEAST
DESTEC,EAST
DOWCHEMICAL,SOUTHWEST
DPNTMTLND,EAST
EAST WINDSOR-G1,SOUTHWEST
EAST WINDSOR-G2,SOUTHWEST
EAST WINDSOR-G3,SOUTHWEST
GREENFIELD ENERGY CENTRE-G1,SOUTHWEST
GREENFIELD ENERGY CENTRE-G2,SOUTHWEST
GREENFIELD ENERGY CENTRE-G3,SOUTHWEST
GREENFIELD ENERGY CENTRE-G4,SOUTHWEST
GREENFIELD SOUTH-G1,SOUTHWEST
GREENFIELD SOUTH-G2,SOUTHWEST
GTAA-G1,WEST
GTAA-G2,WEST
GTAA-G3,WEST
HALTONHILLS-LT.G1,WEST
HALTONHILLS-LT.G2,WEST
HALTONHILLS-LT.G3,WEST
HYDROGEN READY POWER PLANT (HRPP),SOUTHWEST
KAPGS,NORTHEAST
LAKESUPERIOR,NORTHEAST
LENNOX-G1,EAST
LENNOX-G2,EAST
LENNOX-G3,EAST
LENNOX-G4,EAST
NAPANEE-G1,EAST
NAPANEE-G2,EAST
NAPANEE-G3,EAST
NIPIGONGS,NORTHWEST
NORTHBAYGS,NORTHEAST
NPIROQFALLS,NORTHEAST
NPKIRKLAND-G1-G5,NORTHEAST
NPKIRKLAND-G6,NORTHEAST
PORTLANDS-G1,TORONTO
PORTLANDS-G2,TORONTO
PORTLANDS-G3,TORONTO
SITHE GOREWAY-G11,WEST
SITHE GOREWAY-G12,WEST
SITHE GOREWAY-G13,WEST
SITHE GOREWAY-G15,WEST
STCLAIRCGS,SOUTHWEST
TAOHSC,OTTAWA
TASARNIA,SOUTHWEST
TAWINDSOR,SOUTHWEST
THOROLDCGS,NIAGARA
TUNISGS,NORTHEAST
WESTWINDSOR,SOUTHWEST
WHITBYCGS,EAST
YORKCGS-G1,ESSA
YORKCGS-G2,ESSA
ABKENORA,NORTHWEST
AGUASABON,NORTHWEST
ALEXANDER,NORTHWEST
APIROQUOIS,NORTHEAST
ARNPRIOR,OTTAWA
AUBREYFALLS,NORTHEAST
BARRETT,OTTAWA
BECK1,NIAGARA
BECK2,NIAGARA
BECK2 PGS,NIAGARA
CAMERONFALLS,NORTHWEST
CANYON,NORTHEAST
CARIBOUFALLS,NORTHWEST
CARMICHAEL,NORTHEAST
CHATSFALLS,OTTAWA
CHENAUX,OTTAWA
CLERGUE,NORTHEAST
DA WATSON,NORTHEAST
DECEWFALLS,NIAGARA
DECEWND1,NIAGARA
DESJOACHIMS,OTTAWA
EARFALLS,NORTHWEST
FORTFRANCSWC,NORTHWEST
GARTSHORE,NORTHEAST
HARMON,NORTHEAST
HARMON 2,NORTHEAST
HARRIS,NORTHEAST
HOLDEN,OTTAWA
HOLINGSWTH,NORTHEAST
KAKABEKA,NORTHWEST
KIPLING,NORTHEAST
KIPLING 2,NORTHEAST
LITTLELONG,NORTHEAST
LITTLELONG 2,NORTHEAST
LONGSAULTE,EAST
LOWER WHITE RIVER,NORTHEAST
LOWERNOTCH,NORTHEAST
MACKAYGS,NORTHEAST
MANITOUFALLS,NORTHWEST
MISSION,NORTHWEST
MTNCHUTE,OTTAWA
NAGAGAMI,NORTHEAST
OTTERRAPIDS,NORTHEAST
PETER SUTHERLAND SR,NORTHEAST
PINEPORTAGE,NORTHWEST
RAYNER,NORTHEAST
REDROCK,NORTHEAST
SAUNDERS,EAST
SILVERFALLS,NORTHWEST
SMOKY 2,NORTHEAST
STEEPHILL,NORTHEAST
STEWARTVLE,OTTAWA
UMBATAFALLS,NORTHEAST
UPPER WHITE RIVER,NORTHEAST
WELLS,NORTHEAST
WHITEDOG,NORTHWEST
BRUCEA-G1,BRUCE
BRUCEA-G2,BRUCE
BRUCEA-G3,BRUCE
BRUCEA-G4,BRUCE
BRUCEB-G5,BRUCE
BRUCEB-G6,BRUCE
BRUCEB-G7,BRUCE
BRUCEB-G8,BRUCE
DARLINGTON-G1,EAST
DARLINGTON-G2,EAST
DARLINGTON-G3,EAST
DARLINGTON-G4,EAST
PICKERINGB-G5,TORONTO
PICKERINGB-G6,TORONTO
PICKERINGB-G7,TORONTO
PICKERINGB-G8,TORONTO
ARLEN BESS,EAST
GOREWAY BESS,WEST
HAGERSVILLE BESS,NIAGARA
NAPANEE BESS,EAST
ONEIDA ENERGY STORAGE,NIAGARA
TILBURY BATTERY STORAGE,SOUTHWEST
VAUGHAN 1 BESS,ESSA
YORK BESS,ESSA
GRANDSF,NIAGARA
KINGSTONSF,EAST
NANTICOKE SOLAR,NIAGARA
NORTHLAND POWER SOLAR FACILITIES,NORTHEAST
SOUTHGATE SF,BRUCE
STONE MILLS SF,EAST
WINDSOR AIRPORT SF,SOUTHWEST
ADELAIDE,SOUTHWEST
AMARANTH,ESSA
AMHERST ISLAND,EAST
ARMOW,BRUCE
BELLE RIVER,SOUTHWEST
BLAKE,SOUTHWEST
BORNISH,SOUTHWEST
BOW LAKE,NORTHEAST
BOW LAKE 2,NORTHEAST
CEDAR POINT 2,SOUTHWEST
COMBER,SOUTHWEST
CRYSLER,EAST
DILLON,SOUTHWEST
EAST LAKE,SOUTHWEST
ERIEAU,SOUTHWEST
GOSFIELDWGS,SOUTHWEST
GOSHEN,BRUCE
GOULAIS,NORTHEAST
GRAND VALLEY 3,ESSA
GRANDWF,NIAGARA
GREENWICH,NORTHWEST
HENVEY NORTH,NORTHEAST
HENVEY SOUTH,NORTHEAST
JERICHO,SOUTHWEST
K2WIND,BRUCE
KINGSBRIDGE,BRUCE
LANDON,SOUTHWEST
MCLEANSMTNWF-LT.AG_T1,NORTHEAST
NORTH KENT,SOUTHWEST
PAROCHES,EAST
PORT BURWELL,SOUTHWEST
PORTALMA-T1,SOUTHWEST
PORTALMA-T3,SOUTHWEST
PRINCEFARM,NORTHEAST
RAILBEDWF-LT.AG_SR,NORTHEAST
RIPLEY SOUTH,BRUCE
ROMNEY,SOUTHWEST
SANDUSK-LT.AG_T1,BRUCE
SHANNON,NORTHEAST
SPENCE,SOUTHWEST
SUMMERHAVEN,NIAGARA
UNDERWOOD,BRUCE
WEST LINCOLN NRWF,NIAGARA
WOLFE ISLAND,EAST
ZURICH,BRUCE
"""

df_zones_pandas = pd.read_csv(StringIO(generator_zones_data))
spark_zones_df = spark.createDataFrame(df_zones_pandas)

TABLE_ZONES = f"{CATALOG}.{SCHEMA}.generator_zones"
spark_zones_df.write \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(TABLE_ZONES)

print(f"   ✅ Table created: {TABLE_ZONES}")
print(f"   ✅ {len(df_zones_pandas)} generators with zones")

print("\n" + "="*80)
print("✅ IMPORT COMPLETED SUCCESSFULLY")
print("="*80)
print(f"\n📊 Tables created:")
print(f"   1. {OUTPUT_TABLE_DETAIL} - Detailed by generator")
print(f"   2. {OUTPUT_TABLE_AGG} - Aggregated by fuel type")
print(f"   3. {TABLE_ZONES} - Generator zones reference")


In [0]:
# Ajouter la colonne Zone aux tables existantes
from pyspark.sql import functions as F

print("="*80)
print("AJOUT DE LA COLONNE ZONE AUX TABLES")
print("="*80)

# Charger la table de référence des zones
df_zones = spark.table("workspace.energy_forecast.generator_zones")
print(f"\n✅ Table generator_zones chargée: {df_zones.count()} générateurs")

# ============================================================================
# 1. METTRE À JOUR LA TABLE DÉTAILLÉE (BY GENERATOR)
# ============================================================================

print("\n📊 Mise à jour de gen_output_capability_by_generator...")

# Charger la table existante
df_detail = spark.table("workspace.energy_forecast.gen_output_capability_by_generator")

# Faire un LEFT JOIN pour ajouter la colonne Zone
df_detail_aliased = df_detail.alias("detail")
df_zones_aliased = df_zones.alias("zones")

df_detail_with_zone = df_detail_aliased.join(
    df_zones_aliased,
    on="GeneratorName",
    how="left"
)

# Sélectionner explicitement les colonnes
df_detail_final = df_detail_with_zone.select(
    F.col("detail.datetime"),
    F.col("detail.GeneratorName"),
    F.col("detail.FuelType"),
    F.col("zones.Zone"),
    F.col("detail.OutputMW"),
    F.col("detail.CapabilityMW"),
    F.col("detail.UtilizationRate")
)

# Réécrire la table
df_detail_final.write \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.energy_forecast.gen_output_capability_by_generator")

print(f"   ✅ Table mise à jour: gen_output_capability_by_generator")
print(f"   ✅ Lignes: {df_detail_final.count():,}")

# ============================================================================
# 2. CRÉER UNE TABLE AGRÉGÉE PAR ZONE + FUELTYPE
# ============================================================================

print("\n📈 Création de la table agrégée par Zone + FuelType...")

# Agréger par datetime, Zone, FuelType
df_agg_by_zone = df_detail_final.groupBy("datetime", "Zone", "FuelType").agg(
    F.sum("OutputMW").alias("OutputMW"),
    F.sum("CapabilityMW").alias("CapabilityMW"),
    F.count("GeneratorName").alias("NumGenerators")
)

# Calculer le taux d'utilisation
df_agg_by_zone = df_agg_by_zone.withColumn(
    "UtilizationRate",
    F.when(F.col("CapabilityMW") > 0, F.col("OutputMW") / F.col("CapabilityMW")).otherwise(None)
)

# Sauvegarder dans une nouvelle table
df_agg_by_zone.write \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.energy_forecast.gen_output_capability_by_zone")

print(f"   ✅ Table créée: gen_output_capability_by_zone")
print(f"   ✅ Lignes: {df_agg_by_zone.count():,}")

# ============================================================================
# RÉSUMÉ
# ============================================================================

print("\n" + "="*80)
print("✅ COLONNES ZONE AJOUTÉES AVEC SUCCÈS")
print("="*80)
print("\n📊 Tables finales:")
print("   1. gen_output_capability_by_generator - Données détaillées avec Zone")
print("   2. gen_output_capability_by_zone - Agrégé par Zone + FuelType")
print("   3. gen_output_capability - Agrégé par FuelType (global)")
print("   4. generator_zones - Table de référence des zones")